In [ ]:
import cupy as cp

# Small fixed matrices
N = 4
TILE = 2

A = cp.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12],
              [13, 14, 15, 16]], dtype=cp.float32)

B = cp.array([[16, 15, 14, 13],
              [12, 11, 10,  9],
              [ 8,  7,  6,  5],
              [ 4,  3,  2,  1]], dtype=cp.float32)

#  dot product 
C_normal = cp.dot(A, B)

# CUDA tiled kernel
kernel_code = f'''
extern "C" __global__
void matmul_tiled(const float* A, const float* B, float* C, int N) {{
    __shared__ float sA[{TILE}][{TILE}];
    __shared__ float sB[{TILE}][{TILE}];
    
    int row = blockIdx.y * {TILE} + threadIdx.y;
    int col = blockIdx.x * {TILE} + threadIdx.x;
    
    float value = 0.0f;
    
    for (int t = 0; t < (N + {TILE} - 1)/{TILE}; t++) {{
        int tiled_row = row;
        int tiled_col = t*{TILE} + threadIdx.x;
        if(tiled_row < N && tiled_col < N) 
            sA[threadIdx.y][threadIdx.x] = A[tiled_row*N + tiled_col];
        else 
            sA[threadIdx.y][threadIdx.x] = 0.0f;
        
        tiled_row = t*{TILE} + threadIdx.y;
        tiled_col = col;
        if(tiled_row < N && tiled_col < N) 
            sB[threadIdx.y][threadIdx.x] = B[tiled_row*N + tiled_col];
        else 
            sB[threadIdx.y][threadIdx.x] = 0.0f;
        
        __syncthreads();
        
        for(int k=0; k<{TILE}; k++) 
            value += sA[threadIdx.y][k] * sB[k][threadIdx.x];
        __syncthreads();
    }}
    
    if(row < N && col < N) 
        C[row*N + col] = value;
}}
'''

matmul_kernel = cp.RawKernel(kernel_code, 'matmul_tiled')

C_cuda = cp.zeros((N, N), dtype=cp.float32)

threads_per_block = (TILE, TILE)
blocks_per_grid = ((N + TILE - 1)//TILE, (N + TILE - 1)//TILE)

matmul_kernel(blocks_per_grid, threads_per_block, (A, B, C_cuda, N))
cp.cuda.Stream.null.synchronize()

# Print results
print("Matrix A:")
print(A.get())
print("\nMatrix B:")
print(B.get())

print("\n[CuPy dot product result]")
print(C_normal.get())

print("\n[CUDA tiled kernel result]")
print(C_cuda.get())

# Difference
diff = cp.max(cp.abs(C_normal - C_cuda))
print(f"\nMax difference: {diff:.6f}")


Matrix A:
[[ 1.  2.  3.  4.]
 [ 5.  6.  7.  8.]
 [ 9. 10. 11. 12.]
 [13. 14. 15. 16.]]

Matrix B:
[[16. 15. 14. 13.]
 [12. 11. 10.  9.]
 [ 8.  7.  6.  5.]
 [ 4.  3.  2.  1.]]

[CuPy dot product result]
[[ 80.  70.  60.  50.]
 [240. 214. 188. 162.]
 [400. 358. 316. 274.]
 [560. 502. 444. 386.]]

[CUDA tiled kernel result]
[[ 80.  70.  60.  50.]
 [240. 214. 188. 162.]
 [400. 358. 316. 274.]
 [560. 502. 444. 386.]]

Max difference: 0.000000
